# Baseline Models — SVM & Logistic Regression
Point estimates, 5-fold cross-validation with 95 % confidence intervals, and confusion-matrix analysis.

In [1]:
# ── Installs ────────────────────────────────────────────────────────────────
!pip install pandas numpy matplotlib seaborn scikit-learn scipy --quiet

# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy import stats

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Configuration

In [2]:
# ── Paths & constants ────────────────────────────────────────────────────────
DATA_PATH  = "PedroData_Lemmatized_StopwordRemoved.csv"
TEST_SIZE  = 0.2
RANDOM_STATE = 42
CV_FOLDS   = 5
TFIDF_MAX_FEATURES = 10_000
NGRAM_RANGE = (1, 2)

SCORING = {
    "accuracy" : "accuracy",
    "f1"       : "f1",
    "precision": "precision",
    "recall"   : "recall",
}

## Helper Functions

In [3]:
def calculate_ci_95(scores: np.ndarray) -> dict:
    """
    95 % confidence interval via Student's t-distribution
    (appropriate for small n, e.g. 5-fold CV).
    """
    mean      = np.mean(scores)
    std_err   = stats.sem(scores)
    lo, hi    = stats.t.interval(0.95, len(scores) - 1, loc=mean, scale=std_err)
    return {
        "mean"     : mean,
        "ci_lower" : lo,
        "ci_upper" : hi,
        "ci_range" : hi - lo,
        "formatted": f"{mean:.4f}  (95% CI: [{lo:.4f} – {hi:.4f}])",
    }


def evaluate_model(name: str, pipeline, X_train, y_train, X_test, y_test,
                   scoring: dict, cv: int = 5) -> dict:
    """
    Fit *pipeline*, run k-fold CV, evaluate on the held-out test set,
    and return a consolidated result dict.
    """
    # ── fit ──────────────────────────────────────────────────────────────────
    pipeline.fit(X_train, y_train)

    # ── cross-validation ─────────────────────────────────────────────────────
    cv_res = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
    ci = {
        metric: calculate_ci_95(cv_res[f"test_{metric}"])
        for metric in scoring
    }

    # ── test-set predictions ──────────────────────────────────────────────────
    y_pred = pipeline.predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # ── print summary ─────────────────────────────────────────────────────────
    print(f"\n{'═'*60}")
    print(f"  {name}")
    print(f"{'═'*60}")
    print(f"  {'Metric':<12}  {'CV result (5-fold)'}")
    print(f"  {'─'*56}")
    for m, c in ci.items():
        print(f"  {m.capitalize():<12}  {c['formatted']}")
    print(f"\n  Test-set accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"\n{classification_report(y_test, y_pred)}")
    print(f"  Confusion matrix  :\n{cm}\n")

    return {
        "pipeline" : pipeline,
        "ci"       : ci,
        "y_pred"   : y_pred,
        "cm"       : {"TP": tp, "TN": tn, "FP": fp, "FN": fn, "raw": cm},
        "test_acc" : accuracy_score(y_test, y_pred),
    }

## Data Loading & Preprocessing

In [4]:
df = pd.read_csv(DATA_PATH)
print("Shape :", df.shape)
print("Columns:", df.columns.tolist())
print("\nLabel distribution:")
print(df["Label"].value_counts())

Shape : (13459, 3)
Columns: ['Title_filtered_lemma', 'Abstract_filtered_lemma', 'Label']

Label distribution:
Label
0    10711
1     2748
Name: count, dtype: int64


In [5]:
# Concatenate title + abstract into a single text field
df["text"] = (
    df["Title_filtered_lemma"].fillna("") + " " +
    df["Abstract_filtered_lemma"].fillna("")
)

X = df["text"]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Convert to plain lists for compatibility with scikit-learn pipelines
X_train, X_test = X_train.tolist(), X_test.tolist()
y_train, y_test = y_train.tolist(), y_test.tolist()

print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")

Train size: 10767  |  Test size: 2692


## Model Definitions

In [6]:
# Shared TF-IDF settings are pulled from the CONFIG block above.

MODELS = {
    "SVM": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=NGRAM_RANGE,
            max_features=TFIDF_MAX_FEATURES,
        )),
        ("clf", LinearSVC()),
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=NGRAM_RANGE,
            max_features=TFIDF_MAX_FEATURES,
        )),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
}

## Training & Evaluation

In [7]:
results = {}

for name, pipeline in MODELS.items():
    results[name] = evaluate_model(
        name, pipeline,
        X_train, y_train,
        X_test,  y_test,
        scoring=SCORING,
        cv=CV_FOLDS,
    )


════════════════════════════════════════════════════════════
  SVM
════════════════════════════════════════════════════════════
  Metric        CV result (5-fold)
  ────────────────────────────────────────────────────────
  Accuracy      0.8952  (95% CI: [0.8885 – 0.9020])
  F1            0.7335  (95% CI: [0.7173 – 0.7496])
  Precision     0.7635  (95% CI: [0.7387 – 0.7882])
  Recall        0.7061  (95% CI: [0.6834 – 0.7288])

  Test-set accuracy : 0.9008

              precision    recall  f1-score   support

           0       0.93      0.95      0.94      2142
           1       0.78      0.72      0.75       550

    accuracy                           0.90      2692
   macro avg       0.85      0.83      0.84      2692
weighted avg       0.90      0.90      0.90      2692

  Confusion matrix  :
[[2029  113]
 [ 154  396]]


════════════════════════════════════════════════════════════
  Logistic Regression
════════════════════════════════════════════════════════════
  Metric        

## Summary

In [8]:
# ── CV metrics table ─────────────────────────────────────────────────────────
rows = []
for model_name, res in results.items():
    row = {"Model": model_name, "Test Accuracy": f"{res['test_acc']:.4f}"}
    for metric, ci in res["ci"].items():
        row[f"{metric.capitalize()} (mean)"] = f"{ci['mean']:.4f}"
        row[f"{metric.capitalize()} CI"]     = f"[{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]"
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("Model")
print("\n" + "="*100)
print("MODEL COMPARISON — CV Metrics with 95 % Confidence Intervals")
print("="*100)
print(summary_df.to_string())
summary_df.to_csv("model_comparison_results.csv")
print("\nSaved → model_comparison_results.csv")


MODEL COMPARISON — CV Metrics with 95 % Confidence Intervals
                    Test Accuracy Accuracy (mean)       Accuracy CI F1 (mean)             F1 CI Precision (mean)      Precision CI Recall (mean)         Recall CI
Model                                                                                                                                                             
SVM                        0.9008          0.8952  [0.8885, 0.9020]    0.7335  [0.7173, 0.7496]           0.7635  [0.7387, 0.7882]        0.7061  [0.6834, 0.7288]
Logistic Regression        0.8889          0.8852  [0.8773, 0.8931]    0.6704  [0.6483, 0.6926]           0.8102  [0.7801, 0.8403]        0.5719  [0.5507, 0.5931]

Saved → model_comparison_results.csv


In [9]:
# ── Confusion matrix table ────────────────────────────────────────────────────
cm_rows = []
for model_name, res in results.items():
    cm = res["cm"]
    sens = cm["TP"] / (cm["TP"] + cm["FN"]) if (cm["TP"] + cm["FN"]) > 0 else 0
    spec = cm["TN"] / (cm["TN"] + cm["FP"]) if (cm["TN"] + cm["FP"]) > 0 else 0
    cm_rows.append({
        "Model"            : model_name,
        "True Positives"   : cm["TP"],
        "True Negatives"   : cm["TN"],
        "False Positives"  : cm["FP"],
        "False Negatives"  : cm["FN"],
        "Sensitivity (%)"  : f"{sens*100:.2f}",
        "Specificity (%)"  : f"{spec*100:.2f}",
    })

cm_df = pd.DataFrame(cm_rows).set_index("Model")
print("\n" + "="*80)
print("CONFUSION MATRIX COMPARISON")
print("="*80)
print(cm_df.to_string())
cm_df.to_csv("model_confusion_matrices_comparison.csv")
print("\nSaved → model_confusion_matrices_comparison.csv")


CONFUSION MATRIX COMPARISON
                     True Positives  True Negatives  False Positives  False Negatives Sensitivity (%) Specificity (%)
Model                                                                                                                
SVM                             396            2029              113              154           72.00           94.72
Logistic Regression             332            2061               81              218           60.36           96.22

Saved → model_confusion_matrices_comparison.csv
